In [ ]:
%load_ext autoreload
%autoreload 2
from oat.data_module.cosmo_peakweather_dataset import CosmoPeakWeatherConfig, CosmoPeakWeatherDataset
from oat.data_module import transform
from oat.data_module.data_split import create_splits
import matplotlib.pyplot as plt
import yaml


config_path = "/scratch/mch/cdonner/observation_assimilation_transformer/configs/data/cosmo_peakweather_data.yaml"
with open(config_path, 'r') as f:
    config_dict = yaml.safe_load(f)
    
config = CosmoPeakWeatherConfig(**config_dict)

dataset = CosmoPeakWeatherDataset(config)
train_dataset, val_dataset, test_dataset = create_splits(dataset)

In [ ]:
transform = dataset.get_transform(num_data_points=10)

# Investigate data point

In [ ]:
data_point = transform(dataset[0])


for ivar, var in enumerate(config.nwp_vars + config.topo_vars):
    plt.figure(figsize=(8, 6))
    plt.title(f"{var} at final time step")
    im = plt.pcolor(dataset.ref_nwp_ds.x, dataset.ref_nwp_ds.y, 
               data_point['global_data']['global_field'][ivar], cmap='viridis')
    plt.scatter(data_point['observation_data']['coords'][:, 0], data_point['observation_data']['coords'][:, 1],
                c=data_point['observation_data']['obs'][:,-1], label='Stations')
    
    plt.colorbar(im, label=f"{var} value")
    plt.xlabel("X Coordinate")
    plt.show()

In [ ]:
# test iteration speed

import time
start_time = time.time()
observation_list = []
for i in range(100):
    observation_list.append(dataset[i]['observation_data']['obs'][None])
    
end_time = time.time()
print(f"Time taken to iterate over 100 samples: {end_time - start_time} seconds")

In [ ]:
import torch
obs_tensor = torch.cat(observation_list, dim=0)

plt.plot(obs_tensor[:,:, 1])
plt.show()